In [54]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from itertools import groupby

In [55]:
data = pd.read_csv('vit_papers_v3.csv', header = 1)
data.head()
data = data.replace({'NaN': 0, " " : 0, "?" : 0, 'x': 1}).fillna(0)

for col in data.columns:
    try:
        numeric_col = pd.to_numeric(data[col], errors='raise') # attempts to convert the column to numeric, raises an error if it fails
        if (numeric_col % 1 == 0).all(): # checks if all values are integers
            data[col] = numeric_col.astype(int)
        else:
            data[col] = numeric_col
    except (ValueError, TypeError) as e:
        # If conversion fails, print the error message and the column name
        print(f"Could not convert column '{col}' to numeric. Error: {e}")
        print(col)
        pass

data.head()
data.drop(['Fantozzi', 'Kashefi', 'Stassin', 'Unnamed: 33', 'Citation Key'], axis=1, inplace=True)
data = data[data["Year"]>2000]


Could not convert column 'Short Name Identifier' to numeric. Error: Unable to parse string "Transformer Attribution" at position 0
Short Name Identifier
Could not convert column 'Citation Key' to numeric. Error: Unable to parse string "cheferTransformerInterpretabilityAttention2021" at position 0
Citation Key


/tmp/ipykernel_16057/1929764205.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({'NaN': 0, " " : 0, "?" : 0, 'x': 1}).fillna(0)


In [ ]:
data = data[data["Post-hoc"] ==1]
data.drop(["Post-hoc", "Ante-hoc"], axis=1, inplace=True)
data

,Short Name Identifier,Year,Local,Global,Model-specific,Model-class-specific,Model-agnostic,Class-specific,Class-agnostic,Prediction,...,Attention,Activation,Perturbations,Relevance,Gradient,Prototype-based,Concept-based,Weight-Input-Alignment,Masking-based,Connectivity Constraints
2,IA-RED2,2021,1,0,0,1,0,0,1,1,...,0,0,0,0,0,0,0,0,1,0
5,ConceptTransformer,2022,1,0,0,0,1,1,0,1,...,1,0,0,0,0,0,1,0,0,0
6,Patch Interactions,2022,1,0,1,0,0,0,1,0,...,1,0,0,0,0,0,0,0,1,0
8,CAT-XPLAIN,2022,1,0,1,0,0,1,0,1,...,0,0,0,0,0,0,0,0,1,0
9,ViT-NeT,2022,1,0,0,1,0,0,1,1,...,0,1,0,0,0,1,0,0,0,0
10,eX-ViT,2022,1,0,1,0,0,1,0,1,...,1,0,0,0,0,0,0,1,0,0
11,ProtoPFormer,2022,1,1,0,1,0,1,0,1,...,1,1,0,0,0,1,0,0,0,0
14,Bcos-ViT,2023,1,0,1,0,0,1,0,1,...,0,1,0,0,0,0,0,1,0,0
17,ICE,2023,1,0,0,1,0,1,0,1,...,0,1,0,0,0,0,0,0,1,0
19,IA-ViT,2023,1,0,0,0,1,0,1,1,...,1,0,0,0,0,0,0,0,0,0


In [57]:
taxonomy = {"Scope": ['Local', 'Global'],
       'Model Compatability': ['Model-specific', 'Model-class-specific', 'Model-agnostic'],
       'Class-specificity': ['Class-specific', 'Class-agnostic'],
       'Explanation Target': ['Prediction', 'Concept','Embedding', 'Patch Interactions'],
       'Result': [ 'Feature Relevance', 'Scalar', 'Examples', 'Graph', 'Contrastive', 'Text'], # 'Model Component',
       'Functioning': ['Activation', 'Attention', 'Perturbations', 'Relevance', 'Gradient'],
       'Architectural Mechanisms': ['Prototype-based', 'Concept-based', 'Weight-Input-Alignment',
       'Masking-based', 'Connectivity Constraints']}


In [58]:

column_format = "ll|"  # first column (e.g. row labels)

for i, (meta, subgroups) in enumerate(taxonomy.items()):
    column_format += "c" * len(subgroups)

    # add spacing between meta-groups, but not after the last one
    if i < len(taxonomy) - 1:
        #column_format += r"@{\hspace{20pt}}"
        column_format += r"|"
        #column_format +=r"!{\vrule width 0.3pt}"

print(column_format)

ll|cc|ccc|cc|cccc|cccccc|ccccc|ccccc


In [59]:

meta_header = []

for meta, subgroups in taxonomy.items():
    meta_header.append(
        rf"\multicolumn{{{len(subgroups)}}}{{c|}}{{{meta}}}"
    )

meta_header = "&&" +" & ".join(meta_header) 


In [60]:
meta_header

'&&\\multicolumn{2}{c|}{Scope} & \\multicolumn{3}{c|}{Model Compatability} & \\multicolumn{2}{c|}{Class-specificity} & \\multicolumn{4}{c|}{Explanation Target} & \\multicolumn{6}{c|}{Result} & \\multicolumn{5}{c|}{Functioning} & \\multicolumn{5}{c|}{Architectural Mechanisms}'

In [66]:


headers = "&"+" & ".join(
    [r"\rotatebox{90}{" + col + "}" for col in data.columns[1:]]
)



latex = data.replace({0:"", 1:"x"}).to_latex(index=False, header=False, column_format=column_format)


years = data["Year"].values

lines = latex.split("\n")
new_lines = []

for i, line in enumerate(lines):
    #print(line)
    new_lines.append(line)

    i_year = i-3

    
    if i_year < len(years)-1 and years[i_year+1] != years[i_year]:
        #print("hi")
        new_lines.append(r"\addlinespace")

latex = "\n".join(new_lines)

latex = latex.replace(
    r"\toprule",
   "\n" + headers + r"\\" + "\n" +meta_header + r" \\" 
)


print(latex)

\begin{tabular}{ll|cc|ccc|cc|cccc|cccccc|ccccc|ccccc}

&\rotatebox{90}{Year} & \rotatebox{90}{Local} & \rotatebox{90}{Global} & \rotatebox{90}{Model-specific} & \rotatebox{90}{Model-class-specific} & \rotatebox{90}{Model-agnostic} & \rotatebox{90}{Class-specific} & \rotatebox{90}{Class-agnostic} & \rotatebox{90}{Prediction} & \rotatebox{90}{Concept} & \rotatebox{90}{Embedding} & \rotatebox{90}{Patch Interactions} & \rotatebox{90}{Feature Relevance} & \rotatebox{90}{Scalar} & \rotatebox{90}{Examples} & \rotatebox{90}{Graph} & \rotatebox{90}{Text} & \rotatebox{90}{Attention} & \rotatebox{90}{Activation} & \rotatebox{90}{Perturbations} & \rotatebox{90}{Relevance} & \rotatebox{90}{Gradient} & \rotatebox{90}{Prototype-based} & \rotatebox{90}{Concept-based} & \rotatebox{90}{Weight-Input-Alignment} & \rotatebox{90}{Masking-based} & \rotatebox{90}{Connectivity Constraints}\\
&&\multicolumn{2}{c|}{Scope} & \multicolumn{3}{c|}{Model Compatability} & \multicolumn{2}{c|}{Class-specificity} & \mult

In [64]:
lines[:5]

['\\begin{tabular}{ll|cc|ccc|cc|cccc|cccccc|ccccc|ccccc}',
 '\\toprule',
 '\\midrule',
 'IA-RED2 & 2021 & x &  &  & x &  &  & x & x &  &  &  & x &  &  &  &  &  &  &  &  &  &  &  &  & x &  \\\\',
 'ConceptTransformer & 2022 & x &  &  &  & x & x &  & x & x &  &  & x & x &  &  &  & x &  &  &  &  &  & x &  &  &  \\\\']

In [63]:
lines[2]

'\\midrule'